# NYC Taxi Streaming Capstone — Data Profiling Report
This notebook profiles the January 2024 NYC TLC Yellow Taxi dataset and quantifies the validation rules used by the streaming pipeline.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
RAW_PARQUET=Path('data/yellow_tripdata_2024-01.parquet')
ZONE_CSV=Path('data/taxi_zone_lookup.csv')
df=pd.read_parquet(RAW_PARQUET)
print('Rows:',len(df),'Columns:',len(df.columns))
df.head()

## Schema and completeness

In [ ]:
display(pd.DataFrame({'column':df.columns,'dtype':[str(df[c].dtype) for c in df.columns],'null_count':[df[c].isna().sum() for c in df.columns],'null_pct':[100*df[c].isna().mean() for c in df.columns]}))

## Numerical summary

In [ ]:
num=df.select_dtypes(include=np.number).columns
display(df[num].describe(percentiles=[.01,.25,.5,.75,.99]).T)

## Validation-rule profiling

In [ ]:
df['duration_hours']=(pd.to_datetime(df['tpep_dropoff_datetime'])-pd.to_datetime(df['tpep_pickup_datetime'])).dt.total_seconds()/3600
rules={
'negative_fare':df['fare_amount']<0,
'negative_total':df['total_amount']<0,
'zero_distance':df['trip_distance']==0,
'distance_over_200':df['trip_distance']>200,
'passenger_over_8':df['passenger_count']>8,
'passenger_negative':df['passenger_count']<0,
'dropoff_before_pickup':df['tpep_dropoff_datetime']<df['tpep_pickup_datetime'],
'duration_over_12h':df['duration_hours']>12,
'pickup_year_not_2024':pd.to_datetime(df['tpep_pickup_datetime']).dt.year.ne(2024),
'pu_location_invalid':~df['PULocationID'].between(1,265),
'do_location_invalid':~df['DOLocationID'].between(1,265)}
profile=pd.DataFrame([{'rule':k,'violations':int(v.fillna(False).sum()),'rate_pct':100*v.fillna(False).mean()} for k,v in rules.items()])
display(profile.sort_values('violations',ascending=False))

## Duplicate/business-key profiling

In [ ]:
def business_key(r):
 import hashlib
 s='|'.join(str(x) for x in [r['VendorID'],r['tpep_pickup_datetime'],r['PULocationID'],r['DOLocationID'],r['total_amount'],r['fare_amount']])
 return hashlib.md5(s.encode()).hexdigest()
df['trip_key']=df.apply(business_key,axis=1)
print('Duplicate trip_key rows:',int(df['trip_key'].duplicated().sum()))
print('Duplicate (trip_key,pickup) rows:',int(df.duplicated(['trip_key','tpep_pickup_datetime']).sum()))

## Estimated quarantine rate

In [ ]:
invalid_mask=pd.concat([s.rename(k) for k,s in rules.items()],axis=1).any(axis=1)
print(f'Estimated raw validation failure rate: {100*invalid_mask.mean():.2f}%')
print('Note: the streaming producer also injects corrupt messages and duplicate deliveries, so runtime quarantine and deduplication counts will differ from this raw-data estimate.')